In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm

In [3]:
import torch
from transformers import AutoTokenizer, EsmModel
from torch.utils.data import DataLoader
list_num_model=['t6_8M', 't12_35M', 't30_150M', 't33_650M', 't36_3B']


num_model = list_num_model[0]
# 1. Выберите модель (от легкой до тяжелой)
# MODEL_NAME = "facebook/esm2_t6_8M_UR50D"   # Самая быстрая
MODEL_NAME = f"facebook/esm2_{num_model}_UR50D"  # Лучший баланс точности и скорости

print(f"Загрузка модели {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = EsmModel.from_pretrained(MODEL_NAME)
#model = EsmModel.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

/home/admingwi/anaconda3/envs/torch_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Загрузка модели facebook/esm2_t6_8M_UR50D...


Loading weights: 100%|██████████| 107/107 [00:00<00:00, 598.95it/s, Materializing param=encoder.layer.5.output.dense.weight]                      
EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


EsmModel(
  (embeddings): EsmEmbeddings(
    (word_embeddings): Embedding(33, 320, padding_idx=1)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): EsmEncoder(
    (layer): ModuleList(
      (0-5): 6 x EsmLayer(
        (attention): EsmAttention(
          (self): EsmSelfAttention(
            (query): Linear(in_features=320, out_features=320, bias=True)
            (key): Linear(in_features=320, out_features=320, bias=True)
            (value): Linear(in_features=320, out_features=320, bias=True)
            (rotary_embeddings): RotaryEmbedding()
          )
          (output): EsmSelfOutput(
            (dense): Linear(in_features=320, out_features=320, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (LayerNorm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
        )
        (intermediate): EsmIntermediate(
          (dense): Linear(in_features=320, out_features=1280, bias=True)
        )
        (output): EsmOutput(
        

In [4]:
import re
import numpy as np
import pandas as pd

test_seq = "AC-HIK-LMN_"

def clean_peptide(sequence):
    """
    Очищает пептидную последовательность:
    - Удаляет ТОЛЬКО указанные N- и C-терминальные модификации
    - Удаляет скобки и их содержимое
    - Удаляет все дефисы и пробелы
    - Проверяет, что остались только заглавные каноничные аминокислоты
    - Возвращает NaN при любых несоответствиях
    """
    if not isinstance(sequence, str) or pd.isna(sequence):
        return np.nan
    
    seq = sequence.strip()
    
    # 1. Удаляем ТОЛЬКО конкретные N-терминальные модификации
    n_terminal_modifications = ['CH3COO-', 'Ac-']
    
    # Проверяем каждую модификацию (регистрозависимо)
    for mod in n_terminal_modifications:
        if seq.startswith(mod):
            seq = seq[len(mod):]
            break  # Удалили одну модификацию
    
    # 2. Удаляем ТОЛЬКО конкретные C-терминальные модификации
    c_terminal_modifications = ['-NH2', '-NH₂', '-OH', '-COOH', '-CONH2']
    
    # Проверяем каждую модификацию (регистрозависимо)
    for mod in c_terminal_modifications:
        if seq.endswith(mod):
            seq = seq[:-len(mod)]
            break  # Удалили одну модификацию
    
    # 3. Удаляем скобки и их содержимое
    seq = re.sub(r'[\(\[{][^\)\]}]*[\)\]}]', '', seq)
    
    # 4. Удаляем все дефисы и пробелы
    seq = seq.replace('-', '').replace(' ', '')
    
    # 5. Проверки
    if not seq:  # Пустая строка
        return np.nan
    
    if not seq.isupper():  # Есть строчные буквы
        return np.nan
    
    # 6. Проверка на каноничные аминокислоты
    canonical_amino_acids = set('ACDEFGHIKLMNPQRSTVWY')
    for aa in seq:
        if aa not in canonical_amino_acids:
            return np.nan
    
    return seq
    return seq_upper

print("Базовая функция:")
print(f"Вход: '{test_seq}' -> Выход: '{clean_peptide(test_seq)}'")
print()

Базовая функция:
Вход: 'AC-HIK-LMN_' -> Выход: 'nan'



In [21]:
path = 'Data'
folder = 'Toxicity'
data_name = 'NTX'
set_name = 'train'
func_y = 'Tox'

In [19]:
path = 'Data'
folder = 'Pampa'
data_name = 'pampa_b'
set_name = 'train'
func_y = 'PAMPA'
emb_shape = 1280

In [7]:
path = 'Data'
folder = 'Half_life'
data_name = 'Half_life'
set_name = 'train_organ'
#func_y = 'Half-life, h (Homo sapiens)'
func_y = 'Half-life, h'
emb_shape = 1280

In [2]:
path = 'Data'
folder = 'Cell_p'
data_name = 'Cell_p'
set_name = 'test'
func_y = 'CellP'

In [11]:
path = 'Data'
folder = 'Caco'
data_name = 'caco_b'
set_name = 'train'
func_y = 'Caco-2_prm'

In [20]:
hl_df = pd.read_excel(f'{path}/{folder}/{data_name}_{set_name}.xlsx')
hl_df

,sources,Name,Canonical_smiles,PAMPA,Cyclic/Linear (checked),Functional_activity,PK_groups,Sequence
0,CycPeptMPDB,hexa_1045,CC(C)C[C@@H]1NC(=O)[C@@H](C)N(C)C(=O)[C@H](Cc2...,1,Cyclic,NaN,NaN,NaN
1,CycPeptMPDB,hexa_723,CC(C)C[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@H](CC(C...,0,Cyclic,NaN,NaN,NaN
2,CycPeptMPDB,LB05,CCC[C@@H]1NC(=O)CN(CC)C(=O)[C@H](CC(C)C)NC(=O)...,0,Cyclic,NaN,NaN,NaN
3,CycPeptMPDB,hexa_798,CC(C)C[C@H]1C(=O)N[C@@H](Cc2ccccc2)CC(=O)N2CCC...,1,Cyclic,NaN,NaN,NaN
4,CycPeptMPDB,hepta_836,CCCN1CC(=O)N[C@H](CC(C)C)C(=O)N(Cc2ccccc2)CC(=...,0,Cyclic,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
5539,CycPeptMPDB,hexa_975,CC(C)C[C@H]1C(=O)N[C@@H](Cc2ccccc2)CC(=O)N2CCC...,0,Cyclic,NaN,NaN,NaN
5540,CycPeptMPDB,L1_4.3.2.3.3.1,CC(=O)N1CCC[C@@H]1C(=O)N(C)[C@@H](CC(C)C)C(=O)...,0,Cyclic,NaN,NaN,NaN
5541,CycPeptMPDB,L1_9.2.4.3.3.2,CC(=O)N1CCC[C@H]1C(=O)N(C)[C@@H](CC(C)C)C(=O)N...,0,Cyclic,NaN,NaN,NaN
5542,CycPeptMPDB,hepta_149,CC(C)C[C@H]1C(=O)N[C@@H](Cc2ccccc2)C(=O)N2CCC[...,0,Cyclic,NaN,NaN,NaN


In [21]:
df = pd.DataFrame(np.zeros((hl_df.shape[0], 1280)))
df['Sequence']=hl_df.Sequence
df['y']=hl_df[func_y]
num_model = 't33_650M'
df

,0,1,2,3,4,5,6,7,8,9,...,1272,1273,1274,1275,1276,1277,1278,1279,Sequence,y
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,1
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5539,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0
5540,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0
5541,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0
5542,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0


In [22]:
df.to_csv(f'{path}/{folder}/{data_name}_{set_name}_esm2_{num_model}.csv', index=None)

In [9]:
categories = ['Homo sapiens', 'Rattus norvegicus', 'Mus musculus', 'Macaca fascicularis', 'Canis lupus']  # Порядок важен!
mapping = {cat: i for i, cat in enumerate(categories)}
hl_df['Org_ind'] = hl_df['Organism'].map(mapping)

In [36]:
hl_df['Org_ind']=0.0

,sources,Name,Canonical_smiles,"Half-life, h (Homo sapiens)","Half-life, h (Rattus norvegicus)","Half-life, h (Mus musculus)","Half-life, h (Canis lupus)","Half-life, h (Macaca fascicularis)",Cyclic/Linear (checked),Functional_activity,PK_groups,Canonical/Non-canonical,Sequence,Set,Org_ind
0,drugbank,Motixafortide,N=C(N)NCCC[C@H](NC(=O)[C@@H]1CSSC[C@H](NC(=O)[...,2.000,NaN,NaN,NaN,NaN,Cyclic,Haematopoietic growth factors,Группа 2: Системные метаболические регуляторы,"Non-canonical (неканонические ак, D-ак, термин...",4F-benzoyl-RR-Nal-CY-Cit-Kk-PYR-Cit-CR-NH2,Test_Human_Strict,0.0
1,drugbank,Difelikefalin,CC(C)C[C@@H](NC(=O)[C@@H](Cc1ccccc1)NC(=O)[C@H...,27.000,NaN,NaN,NaN,NaN,Linear,Анальгетики,Группа 4: Рецепторно-сигнальные и таргетные мо...,Non-canonical (неканонические ак),fflk-γ-(4-N-piperidinyl)amino carboxylic acid\n,Test_Human_Strict,0.0
2,drugbank,Somatrem,CC(C)[C@H](N)C(=O)N[C@@H](C)C(=O)N[C@H](C(=O)N...,0.325,NaN,NaN,NaN,NaN,Linear,Гормон,Группа 2: Системные метаболические регуляторы,Canonical,VAVGEEPGPR,Test_Human_Strict,0.0
3,drugbank,Dalbavancin,CN[C@H]1C(=O)N[C@@H]2Cc3ccc(cc3)Oc3cc4cc(c3O[C...,346.000,NaN,NaN,NaN,NaN,Cyclic,Антибиотик,Группа 5: Барьерные и локально-ориентированные...,"Non-canonical (неканонические ак, липогликопеп...",NaN,Test_Human_Strict,0.0
4,drugbank,Ziconotide,CSCC[C@@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CCCN...,1.300,NaN,NaN,NaN,NaN,Cyclic,Анальгетики,Группа 4: Рецепторно-сигнальные и таргетные мо...,Canonical,CKGKGAKCSRLMYDCCTGSCRSGKC,Test_Human_Strict,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,Peplife2,WMDF-NH2,CSCC[C@H](NC(=O)[C@@H](N)Cc1c[nH]c2ccccc12)C(=...,0.220,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,WMDF-NH2,Test_Human_Strict,0.0
71,Peplife2,YFLFRPRN-NH2,CC(C)C[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C@@H]...,0.070,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,YFLFRPRN-NH2,Test_Human_Strict,0.0
72,Peplife2,YGGFLRRIRPK-NH2,CC[C@H](C)[C@H](NC(=O)[C@H](CCCNC(=N)N)NC(=O)[...,0.500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,YGGFLRRIRPK-NH2,Test_Human_Strict,0.0
73,Peplife2,YPFF-NH2,NC(=O)[C@H](Cc1ccccc1)NC(=O)[C@H](Cc1ccccc1)NC...,0.100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,YPFF-NH2,Test_Human_Strict,0.0


In [10]:
def get_protein_embeddings(sequences):
    """
    Превращает список пептидных последовательностей в матрицу признаков (эмбеддингов).
    """
    model.eval() # Переключаем модель в режим предсказания (не обучения)
    embeddings = []

    # Обрабатываем последовательности
    with torch.no_grad(): # Отключаем градиенты для экономии памяти
        for seq in sequences:
            clean_seq = clean_peptide(seq)
            if pd.isna(clean_seq):
                embeddings.append(np.zeros(emb_shape))
                continue
            # Токенизация
            inputs = tokenizer(clean_seq, return_tensors="pt", padding=True, truncation=True).to(device)

            # Прогон через модель
            outputs = model(**inputs)

            # outputs.last_hidden_state имеет размерность (1, Seq_Len, Hidden_Dim)
            # Нам нужно получить один вектор на весь пептид.
            # Самый частый метод: усреднение по длине (Mean Pooling)
            # Мы берем среднее по всем аминокислотам, исключая спец-символы (начала и конца)
            token_embeddings = outputs.last_hidden_state

            # Внимание: маска attention нужна, чтобы не усреднять padding (пустоту)
            attention_mask = inputs['attention_mask']

            # Умножаем на маску, чтобы обнулить паддинг, и считаем среднее
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)

            mean_pooling = sum_embeddings / sum_mask

            # Конвертируем в numpy массив и добавляем в список
            embeddings.append(mean_pooling.cpu().numpy()[0])

    return np.array(embeddings)

# --- Пример использования ---
peptides = [
    "ACDEF",           # Короткий пептид
    "MVLSPADKTNVKAA",  # Длинный пептид
    "GGGG"             # Полиглицин
]

print("Генерация эмбеддингов...")
features = np.array(get_protein_embeddings(hl_df.Sequence))

print(f"\nРазмерность матрицы признаков: {features.shape}")


x_df = pd.DataFrame(features)
x_df['Sequence'] = hl_df['Sequence']
#spec_target = np.log10(hl_df[func_y])
#x_df['y'] = spec_target
#x_df['Org_ind'] = hl_df['Org_ind']
x_df['y'] = hl_df['Org_ind']
x_df.to_csv(f'{path}/{folder}/{data_name}_{set_name}_esm2_{num_model}.csv', index=None)

Генерация эмбеддингов...

Размерность матрицы признаков: (889, 1280)


In [38]:
x_df

,0,1,2,3,4,5,6,7,8,9,...,1273,1274,1275,1276,1277,1278,1279,Sequence,y,Org_ind
0,0.005252,0.048175,-0.171349,0.081066,-0.061782,-0.066760,0.132874,0.001083,0.041856,0.065142,...,0.189804,-0.056302,-0.241251,0.021757,-0.235557,-0.076366,-0.087694,SYSMEHFRWGKPVGKKRRPVKVYPNGAEDESAEAFPLEF,-0.602060,0.0
1,0.044234,0.025169,0.105580,0.050535,-0.082038,-0.015482,-0.018064,0.180011,0.079135,0.105745,...,0.122728,-0.006791,0.134502,0.014334,0.001887,-0.055464,-0.033896,KCNTATCATQRLANFLVHSSNNFGPILPPTNVGSNTY-NH2,-0.096910,0.0
2,-0.003458,-0.111105,0.046987,0.107577,-0.029715,-0.130306,0.063758,0.065933,0.207552,0.206298,...,-0.009714,-0.048501,-0.068943,0.088981,-0.098612,0.104779,0.136849,MTPLGPASSLPQSFLLKCLEQVRKIQGDGAALQEKLCATYKLCHPE...,1.672098,0.0
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Hyp-dPhg-wK-Y(Bn)-F,1.079181,0.0
4,-0.024303,-0.000333,-0.023519,0.110215,-0.036727,-0.122598,-0.018346,0.088025,0.036919,0.048873,...,-0.037462,0.009484,0.061497,0.091440,-0.035051,-0.026386,-0.000153,MSYNLLGFLQRSSNFQCQKLLWQLNGRLEYCLKDRMNFDIPEEIKQ...,1.892095,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286,0.020721,0.013345,-0.067855,0.072899,-0.087602,0.004342,-0.076830,0.152373,0.065472,-0.059355,...,0.066286,0.011481,0.153322,0.094468,-0.090317,-0.119199,0.005517,YAEGTFISDYSIAMDKIHQQDFVNWLLAQKGKKNDWKHNITQ,0.778151,0.0
287,0.054229,0.022918,0.031372,-0.020893,-0.041319,-0.045668,-0.161293,0.073997,-0.083483,0.042044,...,0.207303,0.056925,0.019525,0.060205,0.130185,0.016716,-0.005617,YGRKKRRQRRR,0.824126,0.0
288,0.061354,0.075946,0.065789,0.122620,0.005872,-0.087753,-0.134428,0.272418,0.095205,-0.045426,...,0.093366,0.013334,0.148254,-0.049277,-0.128064,0.061513,-0.092456,YPFP-NH2,-0.356547,0.0
289,0.038600,0.060176,0.088626,0.083822,-0.103019,-0.048487,-0.099295,0.213962,0.119494,-0.070947,...,0.113856,0.022454,0.140820,-0.035372,0.047142,0.018137,-0.023592,YVMGHFRWDRFG-NH2,-1.522879,0.0


In [8]:


# 2. Список последовательностей (ВАЖНО: ESM-2 принимает AA-последовательности, не SMILES)
# Если у вас в hl_df строки типа 'MSVPT...', используйте их напрямую.
sequences = hl_df['Sequence'].tolist() 

BATCH_SIZE = 16 # Для 150M модели на 8ГБ VRAM лучше ставить 8-16
dataloader = DataLoader(sequences, batch_size=BATCH_SIZE, shuffle=False)

x = []

with torch.no_grad():
    for batch in tqdm(dataloader, desc="Computing ESM embeddings"):
        # Токенизация пакета
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
        
        # Прогон через модель
        outputs = model(**inputs)
        
        # ESM-2 не имеет [CLS] токена в том же смысле, что BERT. 
        # Самый точный эмбеддинг белка — это среднее по всем аминокислотам (Mean Pooling),
        # исключая токены паддинга.
        
        last_hidden_state = outputs.last_hidden_state
        attention_mask = inputs['attention_mask']
        
        # Усреднение с учетом маски (чтобы нули паддинга не портили среднее)
        mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * mask_expanded, 1)
        sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
        mean_embeddings = (sum_embeddings / sum_mask).cpu().numpy()
        
        x.append(mean_embeddings)

# Объединяем в итоговую матрицу
x = np.vstack(x)
#y = hl_df['Half-life, h'].values

print(f"Готово! Получена матрица: {x.shape}")
x_df = pd.DataFrame(x)
x_df['Sequence'] = hl_df['Sequence']
x_df['y'] = hl_df[func_y]
x_df.to_csv(f'{path}/{folder}/{data_name}_{set_name}_esm2_{num_model}.csv', index=None)

Computing ESM embeddings:   0%|          | 0/19 [00:00<?, ?it/s]


ValueError: Input must be a string, list of strings, or list of ints, got: <class 'torch.Tensor'>

In [13]:
x_df

,0,1,2,3,4,5,6,7,8,9,...,472,473,474,475,476,477,478,479,Sequence,y
0,-0.200533,-0.225300,0.103113,0.115516,-0.026443,-0.061132,-0.347458,0.071054,-0.086484,0.125071,...,-0.020761,0.013623,0.095892,-0.040929,-0.071099,-0.048498,0.005356,0.185675,AAAISCVGSKECLPKCKAQGCKSGKCMNKKCKCYC,1
1,-0.047500,-0.101018,0.110816,0.080631,-0.074042,0.069769,-0.193964,-0.191708,0.040215,0.079199,...,0.037534,0.017536,0.077062,-0.153485,0.096350,-0.180139,-0.054910,0.139136,AACKCDDEGPDIRTAPLTGTVDLGSCNAGWEKCASYYTIIADCCRKKK,1
2,-0.147530,-0.078470,0.148425,0.102237,-0.055279,-0.032635,-0.230466,-0.071518,-0.071318,0.039070,...,0.089020,-0.034297,0.030830,-0.191422,0.047287,-0.177151,0.098004,0.175975,AACLGMFESCDPNNDKCCPNRECNRKHKWCKYKLW,1
3,-0.178812,-0.081433,0.044842,0.022082,0.017096,-0.036496,-0.299995,0.188105,-0.054125,0.074719,...,-0.133406,-0.069889,0.001190,-0.022233,0.006302,0.065803,0.087636,0.123617,AACYSSDCRVKCRAMGFSSGKCIDSKCKCYK,1
4,-0.176491,-0.080208,0.053259,0.027016,0.031920,-0.051085,-0.311547,0.170977,-0.062985,0.083411,...,-0.137693,-0.047540,-0.013108,-0.022528,-0.005671,0.048149,0.092309,0.115683,AACYSSDCRVKCVAMGFSSGKCINSKCKCYK,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2127,-0.037056,-0.199642,0.198604,0.019158,0.122124,0.076083,-0.339653,0.066412,0.111547,-0.038848,...,-0.070514,-0.084567,-0.051785,-0.054044,-0.064725,-0.064109,-0.061861,0.209373,MTSFKIVIVCLALLVAVASARSRDMMSDDERDYHFSKRGIPCACDS...,1
2128,0.034889,-0.147952,0.197030,0.128665,0.189649,-0.032202,-0.233049,0.149725,0.048636,-0.006775,...,0.052063,-0.095520,-0.171956,-0.125118,-0.006887,-0.022775,-0.000892,0.136854,MHLSLARSAVLMLLLLFALGNFVVVQSGQITRDVDNGQLTDNRRNL...,1
2129,-0.169491,-0.130069,0.273306,0.089602,0.070233,0.036546,-0.328054,0.044491,0.013111,-0.051233,...,0.093282,0.105279,-0.035602,-0.189991,-0.059259,-0.042921,0.033676,0.168247,MKQYIFFLALIVLVSTFAEAGKKTEILDKVKKVFSKAKDKILAGVE...,1
2130,-0.034375,-0.118108,0.155442,0.059613,0.082293,-0.018995,-0.208287,0.075858,-0.048873,0.032613,...,-0.090487,-0.132756,-0.123633,-0.199197,0.175325,0.005465,0.029835,0.181662,MKTLLLALVVLAFVCLGSADQVGLGKEQIDRGRRQAIGPPFTRCSQ...,1
